# Speculative decoding from scratch

Walkthrough for **ai-learn-25-speculative-decoding**.

A cheap **draft** LM proposes gamma tokens; a stronger **target** LM verifies them in one parallel forward;
we accept a prefix with `min(1, q/p)` and resample from the residual on rejection (Leviathan / Chen).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
from corpus import build_corpus, encode, make_maps, decode
from lm import train_ngram, vanilla_decode, mix_probs
from speculate import speculative_decode, speculative_step, compare_distributions

SEED = 42
text, chars = build_corpus(repeats=40)
stoi, itos = make_maps(chars)
ids = encode(text, stoi)
V = len(chars)
split = int(0.9 * len(ids))
train_ids, held = ids[:split], ids[split:]
draft = train_ngram(train_ids, order=2, V=V, cost_per_token=1.0)
target = train_ngram(train_ids, order=3, V=V, cost_per_token=8.0)
print('vocab', V, 'train chars', len(train_ids))
print('draft ppl', round(draft.perplexity(held[:500]), 4),
      'target ppl', round(target.perplexity(held[:500]), 4))


## One speculative step
Draft samples gamma tokens, target scores them, accept with `min(1, q/p)`.


In [ ]:
prompt = held[10:18]
rng = np.random.default_rng(SEED)
new_toks, stats = speculative_step(draft, target, prompt, gamma=4, rng=rng)
print('prompt :', repr(decode(prompt, itos)))
print('emitted:', repr(decode(new_toks, itos)), 'stats', {k: round(v, 4) for k, v in stats.items()})


## gamma sweep vs vanilla (FLOP proxy)
Vanilla cost = `n_tokens * target_cost`. Speculative ~= `#steps * (gamma * draft_cost + target_cost)`.


In [ ]:
n_tokens, n_prompts = 32, 12
prompts = [held[i:i+8] for i in range(0, n_prompts * 9, 9)]
vanilla_flops = 0.0
for i, p in enumerate(prompts):
    _, st = vanilla_decode(target, p, n_tokens, np.random.default_rng(SEED + 1000 + i))
    vanilla_flops += st['flops']

rows = []
for gamma in [1, 2, 3, 4, 5, 6, 8]:
    flops, accs = 0.0, []
    for i, p in enumerate(prompts):
        _, st = speculative_decode(draft, target, p, n_tokens, gamma,
                                   np.random.default_rng(SEED + 2000 + i), draft_mix=1.0)
        flops += st['flops']
        accs.append(st['mean_acceptance_rate'])
    rows.append((gamma, round(float(np.mean(accs)), 4), round(vanilla_flops / flops, 4)))
print('gamma | accept | speedup')
for g, a, s in rows:
    print(f'{g:2d} | {a:.4f}  | {s:.4f}x')


## Law check: perfect draft (mix=0) -> acceptance = 1
Mixing draft probs toward the target (`draft_mix->0`) makes p == q, so every draft token is accepted.


In [ ]:
ctxs = [held[i:i+4] for i in range(0, 100, 5)]
for mix in [1.0, 0.5, 0.0]:
    d = compare_distributions(draft, target, ctxs, draft_mix=mix)
    flops, accs = 0.0, []
    for i, p in enumerate(prompts):
        _, st = speculative_decode(draft, target, p, n_tokens, 4,
                                   np.random.default_rng(SEED + 3000 + i), draft_mix=mix)
        flops += st['flops']; accs.append(st['mean_acceptance_rate'])
    print(f'mix={mix:.1f}  TV={d["mean_tv"]:.4f}  accept={np.mean(accs):.4f}  speedup={vanilla_flops/flops:.4f}x')


## Takeaways
1. Speculative decoding preserves the target distribution exactly via accept/reject + residual sampling.
2. Speedup comes from verifying gamma draft tokens in **one** target forward.
3. Acceptance (and therefore speedup) tracks how close the draft is to the target.
4. Run `python run_smoke.py` from the repo root for the committed `results/` add-on.
